In [ ]:
!pip uninstall -y torchao
!pip install open_clip_torch peft timm transformers -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import copy
import time
import random
import warnings
import numpy as np
import pandas as pd
!pip install open_clip_torch
!pip install peft

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as models

import open_clip

from peft import LoraConfig, get_peft_model

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using Device:', device)

Using Device: cuda


In [ ]:
dataset_path = '/content/drive/MyDrive/SAM_Masked_1500'

classes = sorted([
    cls for cls in os.listdir(dataset_path)
    if os.path.isdir(os.path.join(dataset_path, cls))
])

print('Classes:', classes)

Classes: ['Cyst', 'Normal', 'Stone', 'Tumor']


In [ ]:
all_filepaths = []
all_labels = []

valid_extensions = ('.png', '.jpg', '.jpeg', '.bmp')

for cls in classes:
    class_dir = os.path.join(dataset_path, cls)

    for file in os.listdir(class_dir):
        if file.lower().endswith(valid_extensions):
            all_filepaths.append(os.path.join(class_dir, file))
            all_labels.append(cls)


df = pd.DataFrame({
    'filepath': all_filepaths,
    'label': all_labels
})

print('\nTotal Images:', len(df))
print(df['label'].value_counts())

# ============================================================
# LABEL ENCODING
# ============================================================

class_to_idx = {cls:i for i, cls in enumerate(classes)}
idx_to_class = {i:cls for cls, i in class_to_idx.items()}


Total Images: 1498
label
Normal    611
Cyst      447
Tumor     275
Stone     165
Name: count, dtype: int64


In [ ]:
class KidneyDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        img_path = self.df.loc[idx, 'filepath']
        label = self.df.loc[idx, 'label']

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        label = class_to_idx[label]

        return image, label

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
class HybridModel(nn.Module):

    def __init__(self):
        super(HybridModel, self).__init__()

        clip_model, _, _ = open_clip.create_model_and_transforms(
            'ViT-B-32',
            pretrained='openai'
        )

        lora_config = LoraConfig(
            r=8,
            lora_alpha=16,
            target_modules=[
                'out_proj',
                'c_fc',
                'c_proj'
            ],
            lora_dropout=0.1,
            bias='none'
        )

        clip_model.visual = get_peft_model(
            clip_model.visual,
            lora_config
        )

        for param in clip_model.visual.parameters():
            param.requires_grad = False

        for name, param in clip_model.visual.named_parameters():
            if 'lora' in name.lower():
                param.requires_grad = True

        self.clip_visual = clip_model.visual

        self.cnn_model = models.efficientnet_b0(weights='DEFAULT')

        cnn_features = self.cnn_model.classifier[1].in_features
        self.cnn_model.classifier = nn.Identity()

        self.fc = nn.Sequential(
            nn.Linear(512 + cnn_features, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, len(classes))
        )

    def forward(self, x):

        clip_features = self.clip_visual(x)
        cnn_features = self.cnn_model(x)

        combined = torch.cat([
            clip_features,
            cnn_features
        ], dim=1)

        output = self.fc(combined)

        return output, combined

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):

    model.train()

    running_loss = 0

    preds_all = []
    labels_all = []

    for images, labels in tqdm(loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            outputs, _ = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        preds = torch.argmax(outputs, dim=1)

        preds_all.extend(preds.cpu().numpy())
        labels_all.extend(labels.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    epoch_acc = accuracy_score(labels_all, preds_all)

    return epoch_loss, epoch_acc


In [ ]:
def evaluate(model, loader, criterion):

    model.eval()

    running_loss = 0

    preds_all = []
    labels_all = []
    features_all = []

    with torch.no_grad():

        for images, labels in tqdm(loader):

            images = images.to(device)
            labels = labels.to(device)

            outputs, features = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            preds = torch.argmax(outputs, dim=1)

            preds_all.extend(preds.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
            features_all.extend(features.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    epoch_acc = accuracy_score(labels_all, preds_all)

    return (
        epoch_loss,
        epoch_acc,
        labels_all,
        preds_all,
        features_all
    )

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

In [ ]:
all_fold_metrics = []
all_true_labels = []
all_pred_labels = []
all_features = []

combined_history = {
    'train_loss': [],
    'test_loss': [],
    'train_acc': [],
    'test_acc': []
}

In [ ]:
for fold, (train_idx, test_idx) in enumerate(
    skf.split(df['filepath'], df['label'])
):

    print('\n' + '='*60)
    print(f'FOLD {fold+1}/5')
    print('='*60)

    train_df = df.iloc[train_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

    print('\nTrain Distribution:')
    print(train_df['label'].value_counts())

    print('\nTest Distribution:')
    print(test_df['label'].value_counts())

    train_dataset = KidneyDataset(
        train_df,
        transform=train_transform
    )

    test_dataset = KidneyDataset(
        test_df,
        transform=test_transform
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    model = HybridModel().to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )

    scaler = torch.amp.GradScaler('cuda')

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0

    fold_history = {
        'train_loss': [],
        'test_loss': [],
        'train_acc': [],
        'test_acc': []
    }

    # ========================================================
    # EPOCH LOOP
    # ========================================================

    for epoch in range(10):

        print(f'\nEpoch {epoch+1}/10')

        start_time = time.time()

        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            scaler
        )

        (
            test_loss,
            test_acc,
            y_true,
            y_pred,
            features
        ) = evaluate(
            model,
            test_loader,
            criterion
        )

        fold_history['train_loss'].append(train_loss)
        fold_history['test_loss'].append(test_loss)
        fold_history['train_acc'].append(train_acc)
        fold_history['test_acc'].append(test_acc)

        print(f'Train Loss: {train_loss:.4f}')
        print(f'Train Accuracy: {train_acc:.4f}')
        print(f'Test Loss: {test_loss:.4f}')
        print(f'Test Accuracy: {test_acc:.4f}')

        elapsed = time.time() - start_time
        print(f'Time: {elapsed:.2f} sec')

        if test_acc > best_acc:
            best_acc = test_acc
            best_model_wts = copy.deepcopy(model.state_dict())

    # ========================================================
    # LOAD BEST MODEL
    # ========================================================

    model.load_state_dict(best_model_wts)

    (
        test_loss,
        test_acc,
        y_true,
        y_pred,
        features
    ) = evaluate(
        model,
        test_loader,
        criterion
    )

    # ========================================================
    # METRICS
    # ========================================================

    cm = confusion_matrix(y_true, y_pred)

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average='macro'
    )

    sensitivity = recall_score(
        y_true,
        y_pred,
        average='macro'
    )

    specificity_per_class = []

    for i in range(len(classes)):

        tp = cm[i, i]

        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)

        specificity = tn / (tn + fp + 1e-8)

        specificity_per_class.append(specificity)

    specificity = np.mean(specificity_per_class)

    print('\nFold Metrics')
    print('-'*40)
    print(f'Accuracy    : {accuracy:.4f}')
    print(f'Precision   : {precision:.4f}')
    print(f'Sensitivity : {sensitivity:.4f}')
    print(f'Specificity : {specificity:.4f}')

    all_fold_metrics.append({
        'Fold': fold + 1,
        'Accuracy': accuracy,
        'Precision': precision,
        'Sensitivity': sensitivity,
        'Specificity': specificity
    })

    all_true_labels.extend(y_true)
    all_pred_labels.extend(y_pred)
    all_features.extend(features)

    combined_history['train_loss'].append(fold_history['train_loss'])
    combined_history['test_loss'].append(fold_history['test_loss'])
    combined_history['train_acc'].append(fold_history['train_acc'])
    combined_history['test_acc'].append(fold_history['test_acc'])



FOLD 1/5

Train Distribution:
label
Normal    489
Cyst      357
Tumor     220
Stone     132
Name: count, dtype: int64

Test Distribution:
label
Normal    122
Cyst       90
Tumor      55
Stone      33
Name: count, dtype: int64

Epoch 1/10


100%|██████████| 10/10 [00:43<00:00,  4.37s/it]


Train Loss: 1.0663
Train Accuracy: 0.5826
Test Loss: 0.8274
Test Accuracy: 0.6633
Time: 105.63 sec

Epoch 2/10


100%|██████████| 10/10 [00:02<00:00,  3.57it/s]


Train Loss: 0.5999
Train Accuracy: 0.7713
Test Loss: 0.3699
Test Accuracy: 0.8767
Time: 16.46 sec

Epoch 3/10


...

Train Loss: 0.0507
Train Accuracy: 0.9842
Test Loss: 0.0554
Test Accuracy: 0.9833
Time: 17.09 sec

Epoch 10/10


100%|██████████| 10/10 [00:02<00:00,  3.44it/s]


Train Loss: 0.0377
Train Accuracy: 0.9900
Test Loss: 0.0534
Test Accuracy: 0.9799
Time: 17.45 sec


100%|██████████| 10/10 [00:02<00:00,  3.43it/s]


Fold Metrics
----------------------------------------
Accuracy    : 0.9833
Precision   : 0.9819
Sensitivity : 0.9729
Specificity : 0.9941


In [ ]:
metrics_df = pd.DataFrame(all_fold_metrics)

print('\n' + '='*60)
print('FINAL CROSS VALIDATION RESULTS')
print('='*60)

print(metrics_df)

print('\nAverage Metrics')
print('-'*40)
print(f"Mean Accuracy    : {metrics_df['Accuracy'].mean():.4f}")
print(f"Mean Precision   : {metrics_df['Precision'].mean():.4f}")
print(f"Mean Sensitivity : {metrics_df['Sensitivity'].mean():.4f}")
print(f"Mean Specificity : {metrics_df['Specificity'].mean():.4f}")



FINAL CROSS VALIDATION RESULTS
   Fold  Accuracy  Precision  Sensitivity  Specificity
0     1  0.983333   0.973698     0.979545     0.994596
1     2  0.990000   0.993220     0.983333     0.996001
2     3  0.973333   0.976455     0.958018     0.990167
3     4  0.956522   0.956546     0.945050     0.983581
4     5  0.983278   0.981913     0.972949     0.994076

Average Metrics
----------------------------------------
Mean Accuracy    : 0.9773
Mean Precision   : 0.9764
Mean Sensitivity : 0.9678
Mean Specificity : 0.9917
